In [1]:
from pyspark.sql import SparkSession
import os

# Nạp đầy đủ Package cho Kafka, Iceberg, Nessie và S3 (MinIO)
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1,'
    'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,'
    'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.67.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4 '  # THƯ VIỆN CÒN THIẾU Ở ĐÂY
    'pyspark-shell'
)

spark = SparkSession.builder \
    .appName("Fix-S3A-Error") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://silver/") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. Đọc bảng bằng Spark SQL
df = spark.sql("SELECT * FROM nessie.silver_db.amazon_purchase_silver")

# 3. Chuyển đổi sang Pandas và hiển thị (đẹp hơn show())
# Lấy 10 dòng đầu tiên để xem nhanh
pdf = df.limit(5).toPandas()

In [2]:
pdf

,order_date,unit_price,quantity,state,product_title,product_code,category,survey_id,processed_at
0,2019-05-08,7.19,1.0,MO,Majic 4-Pieces 4-Sizes Plastic Funnel Set for ...,B00OABYBJG,FUNNEL,R_3CZ8DrIEalrYlgu,2026-03-11 10:44:37.218
1,2019-05-08,8.40,1.0,PA,"Grove Square Cappuccino Pods, French Vanilla, ...",B005K4Q1YA,COFFEE,R_3PAyJj7mQvMzMco,2026-03-11 10:44:37.218
2,2019-05-08,15.46,1.0,VA,Solo 418 One-Hand Pressure Sprayer with Ergono...,B000BX4VXI,PUMP_SPRAYER,R_sL3ziBSWCD3aQ93,2026-03-11 10:44:37.218
3,2019-05-08,8.99,1.0,MI,VSUDO 2 Pairs Double Layer Flat Sneaker Shoe L...,B07GDS3JPP,SHOELACE,R_1l4zCtGUTvAWdS6,2026-03-11 10:44:37.218
4,2019-05-08,14.97,1.0,VA,AhfuLife Wooden Elephant Cell Phone Holder/Sta...,B078ZVJJ4J,CADDY,R_1pxM7P5tV3JinW2,2026-03-11 10:44:37.218


In [3]:
from pyspark.sql.functions import count

# Tổng số dòng
total_count = df.count()

# Số dòng distinct
distinct_count = df.distinct().count()

print("Total rows:", total_count)
print("Distinct rows:", distinct_count)
print("Duplicate rows:", total_count - distinct_count)

Total rows: 1670103
Distinct rows: 1670103
Duplicate rows: 0


In [4]:
from pyspark.sql.functions import col

duplicates = (
    df.groupBy(df.columns)
      .count()
      .filter(col("count") > 1)
      .orderBy(col("count").desc())
)

# Chỉ lấy 50 dòng đầu rồi mới chuyển
pdf = duplicates.limit(10).toPandas()

pdf

,order_date,unit_price,quantity,state,product_title,product_code,category,survey_id,processed_at,count
